# Tầng 2 — PhoBERT phân loại câu (kiểu BERTSum)

Notebook này **huấn luyện** một bộ chọn câu có giám sát: PhoBERT mã hoá cả bài một
lần, mỗi câu được cho một điểm, rồi lấy 3 câu điểm cao nhất làm bản tóm tắt.

## Tầng này trả lời câu hỏi gì

Tầng 0-1 chọn câu mà không học gì. Oracle-3 cho thấy trần của việc chọn câu là
**48,09** trong khi Lead-3 chỉ **27,45** — tức khoảng **21 điểm** nằm trong tầm với
chỉ nhờ chọn câu khéo hơn. Tầng 2 đo xem một bộ chọn *học được* lấy lại bao nhiêu
trong 21 điểm ấy.

## Ràng buộc 256 token, và vì sao vẫn chấp nhận nó

PhoBERT chỉ có 256 vị trí. Đo trên 300 bài `val` bằng chính tokenizer PhoBERT:
**80% số bài vượt 256 token**, một cửa sổ chỉ nhìn thấy trung bình **60,6% số câu**.
Hệ quả: trần của tầng này không phải 49,8 mà là **44,4** — Oracle-3 bị giới hạn trong
đúng cửa sổ ấy.

Vẫn chấp nhận cho bản đầu, vì 44,4 còn hơn Lead-3 khoảng **16 điểm**, tức thừa dư địa
để tầng 2 có nghĩa. Không làm cửa sổ trượt ngay: đó là thêm một biến chưa đo vào một
tầng chưa chạy. Con số 44,4 được báo cáo như một **trần thứ hai**, và chính khoảng
cách 44,4 so với 49,8 là một kết quả — nó đo cái giá của việc chọn một bộ mã hoá chỉ
đọc được 256 token.

## Hai quyết định đã chốt

**Nhãn lấy từ `oracle_indices()`**, đúng hàm đã dựng trần extractive ở tầng 0. Nếu
nhãn và trần đến từ hai cách tính khác nhau thì khoảng cách giữa tầng 2 và Oracle-3
không còn đọc được.

**Đầu vào là dạng TÁCH TỪ**, vì PhoBERT được pretrain trên văn bản đã tách từ. Đầu ra
vì thế cũng ở dạng tách từ, giống hệt tầng 0-1; khâu chấm điểm tự quy về dạng thô.

## Trước khi chạy: Settings

| Mục | Đặt thành |
|---|---|
| **Accelerator** | `GPU T4 x2` |
| **Internet** | `On` (để tải `vinai/phobert-base`) |

Notebook này **không** cần gắn output của notebook nào khác: nó huấn luyện từ PhoBERT
gốc trên Hub, nên `kernel_sources` để rỗng. Nhờ vậy nó cũng không đụng gì tới
checkpoint BARTpho mà tầng 4 đang cần.


In [ ]:
# ==== CHI SUA O NAY ===================================================
MODEL = "vinai/phobert-base"
TRAIN_SPLIT = "train_20k"
EVAL_SPLIT = "val"          # KHONG doi thanh test
EPOCHS = 3
LR = 2e-5
BATCH = 8
MAX_LEN = 256               # gioi han vi tri cua PhoBERT, khong phai lua chon
# ======================================================================

REPO = "https://github.com/ICY825/SummariseVietNamese.git"
DIR = "/kaggle/working/BTL_DL"
OUT = "/kaggle/working/tang2"


def check(code, what):
    """Dung notebook neu lenh `!` ngay truoc do loi — `!lenh` loi khong nem ngoai le."""
    if code != 0:
        raise RuntimeError(f"{what} THAT BAI (ma thoat {code}) -- xem log ngay tren.")
    print(f"{what}: OK")


print(f"Tang 2: {MODEL} tren {TRAIN_SPLIT}, cham tren {EVAL_SPLIT}, cua so {MAX_LEN} token")


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import urllib.request
import torch
print("GPU thay duoc:", torch.cuda.device_count())
if torch.cuda.device_count() == 0:
    raise RuntimeError("Khong thay GPU. Settings > Accelerator > GPU T4 x2.")
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise RuntimeError("Khong ra duoc Internet. Settings > Internet > On.") from e
print("Moi truong: OK")

In [ ]:
import os
if not os.path.isdir(DIR):
    !git clone -q {REPO} {DIR}
    check(_exit_code, "git clone")
os.chdir(DIR)
!git pull -q
check(_exit_code, "git pull")
!git log --oneline -1
print("cwd:", os.getcwd())

## Huấn luyện và chấm điểm

Một lần gọi `phobert_sent.py`: dựng nhãn oracle, huấn luyện, sinh bản tóm tắt trên `val`, chấm điểm, rồi ghi bảng chỉ số cùng `run.json` vào `results/`.


In [ ]:
import time

t0 = time.time()
cmd = (f"CUDA_VISIBLE_DEVICES=0 python src/models/phobert_sent.py "
       f"--model {MODEL} --train-split {TRAIN_SPLIT} --eval-split {EVAL_SPLIT} "
       f"--epochs {EPOCHS} --lr {LR} --batch {BATCH} --max-len {MAX_LEN} --out {OUT}")
print(cmd)
!{cmd}
check(_exit_code, "huan luyen + cham tang 2")
print(f"\nXong trong {(time.time() - t0) / 60:.1f} phut.")


In [ ]:
# Gom ket qua (khong gom trong so) thanh mot zip de tai ve tu tab Output.
import pathlib
import zipfile

picked = sorted(p for p in pathlib.Path("results").rglob("*.json")
                if p.name.startswith("phobert-sent-"))
if not picked:
    raise RuntimeError("Khong thay file ket qua nao cua tang 2.")
zpath = "/kaggle/working/ket_qua_tang2.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in picked:
        z.write(p, p.as_posix())
        print(f"  {p.as_posix():86s} {p.stat().st_size / 1e6:6.2f} MB")
print("Da dong goi:", zpath)
